In [5]:
import unittest

from road_network import (
    RoadNetwork,
    RoadSegment,
    generate_road_network,)

class TestRoadNetwork(unittest.TestCase):

    def setUp(self):
        self.network = generate_road_network(
            num_nodes=50,
            num_segments=80,
            seed=42,
        )

    def test_correct_network_size(self):
        self.assertEqual(len(self.network.nodes), 50)
        self.assertEqual(len(self.network.segments), 80)

    def test_network_is_connected(self):
        self.assertTrue(self.network.is_connected())

    def test_network_is_valid(self):
        self.assertTrue(self.network.validate())

    def test_segment_ids_are_unique(self):
        ids = list(self.network.segments.keys())
        self.assertEqual(len(ids), len(set(ids)))

    def test_all_road_classes_are_present(self):
        for road_class in (
            "highway", "main_road", "small_road"
        ):
            self.assertGreater(
                len(self.network.get_segments_by_class(road_class)),
                0,
            )

    def test_roughness_is_bounded(self):
        for segment in self.network.segments.values():
            self.assertTrue(
                0 <= segment.initial_roughness <= 1
            )
            self.assertTrue(
                0 <= segment.true_roughness <= 1
            )

    def test_bidirectional_adjacency(self):
        segment = self.network.get_segment("R00001")

        self.assertIn(
            (segment.end_node, segment.segment_id),
            self.network.get_neighbors(segment.start_node),
        )

        self.assertIn(
            (segment.start_node, segment.segment_id),
            self.network.get_neighbors(segment.end_node),
        )

    def test_same_seed_produces_same_network(self):
        other = generate_road_network(
            num_nodes=50,
            num_segments=80,
            seed=42,
        )

        self.assertEqual(
            list(self.network.segments.values()),
            list(other.segments.values()),
        )

    def test_impossible_network_is_rejected(self):
        with self.assertRaises(ValueError):
            generate_road_network(
                num_nodes=10,
                num_segments=5,
            )

    def test_duplicate_segment_is_rejected(self):
        segment = self.network.get_segment("R00001")

        with self.assertRaises(ValueError):
            self.network.add_segment(segment)


unittest.main(argv=["first-arg-is-ignored"], exit=False)


..........
----------------------------------------------------------------------
Ran 10 tests in 0.029s

OK
